# Import sempy and Fabric libraries

In [2]:
import sempy
import sempy.fabric as fabric
import pandas

StatementMeta(, , -1, Waiting, , Waiting)

## Import Parameters

In [6]:
%run 2. Parameters

StatementMeta(, bfdae0b4-e3e0-49d4-a6d9-73fec5048125, 10, Finished, Available, Finished)

## Setup Workspace and Dataset, then Run DAX Query

In [7]:
tables_df = fabric.evaluate_dax(datasetId, dax_string=dax_query_tables, workspace=workspaceId)
tables_df.columns = [col.strip("[]").upper() for col in tables_df.columns]
spark_tables_df = spark.createDataFrame(tables_df)
display(spark_tables_df)

SynapseWidget(Synapse.DataFrame, 811be681-fec4-49a6-a437-49a7ec3046c8)

In [8]:
# Write to lakehouse table
table_name = "TABLES"
# Convert to Spark DataFrame and write

spark_tables_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_name)

StatementMeta(, bfdae0b4-e3e0-49d4-a6d9-73fec5048125, 12, Finished, Available, Finished)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: a72e5f5f-46a3-46cc-a3c1-2d5d4d75c23a).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- ID: long (nullable = true)
-- MODELID: long (nullable = true)
-- NAME: string (nullable = true)
-- DATACATEGORY: string (nullable = true)
-- DESCRIPTION: string (nullable = true)
-- ISHIDDEN: boolean (nullable = true)
-- TABLESTORAGEID: long (nullable = true)
-- MODIFIEDTIME: timestamp (nullable = true)
-- STRUCTUREMODIFIEDTIME: timestamp (nullable = true)
-- SYSTEMFLAGS: long (nullable = true)
-- SHOWASVARIATIONSONLY: boolean (nullable = true)
-- ISPRIVATE: boolean (nullable = true)
-- DEFAULTDETAILROWSDEFINITIONID: long (

In [5]:
columns_df = fabric.evaluate_dax(datasetId, dax_string=dax_query_columns, workspace=workspaceId)
columns_df.columns = [col.strip("[]").upper() for col in columns_df.columns]
spark_columns_df = spark.createDataFrame(columns_df)
display(spark_columns_df)

StatementMeta(, 7bbd120f-3f46-4b31-b27f-983d49dfe7f5, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 01600924-f1ab-44a9-9413-907c1c6c144b)

In [6]:
# Set the datetime rebase mode
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# Write to lakehouse table
table_name = "COLUMNS"
# Convert to Spark DataFrame and write
spark_columns_df.write.mode("overwrite").format("delta").saveAsTable(table_name)

StatementMeta(, 7bbd120f-3f46-4b31-b27f-983d49dfe7f5, 10, Finished, Available, Finished)

## List Columns

In [ ]:
from sempy import fabric

# Get the metadata including data types
list_columns_df = fabric.list_columns(dataset=datasetId, workspace=workspaceId)

# Add columns to your pandas DataFrame
list_columns_df['workspace_id'] = workspaceId
list_columns_df['dataset_id'] = datasetId
list_columns_df['database_name'] = semantic_model_name

# Automatically replace spaces with underscores in all column names
list_columns_df.columns = list_columns_df.columns.str.replace(' ', '_').str.upper()

# Drop columns with void/null datatypes
list_columns_df = list_columns_df.dropna(axis=1, how='all')

# Alternatively, explicitly drop the problematic column
# list_columns_df = list_columns_df.drop(columns=['sort_by_column'], errors='ignore')

# Define table name
table_name = "list_columns"

# Drop the table if it exists to clear any metadata issues
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Convert to Spark DataFrame and write
spark_list_columns_df = spark.createDataFrame(list_columns_df)
spark_list_columns_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(table_name)

print(f"Successfully wrote {spark_list_columns_df.count()} rows to table '{table_name}'")

# Now query and display
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1000")
display(df)

StatementMeta(, c8e4c7c9-9e6d-46c8-86ba-e8f850e91ded, 44, Finished, Available, Finished)

Successfully wrote 31 rows to table 'list_columns'


SynapseWidget(Synapse.DataFrame, b185a4f2-856c-46cd-874f-853981bb0f89)

In [15]:
df = spark.sql("SELECT * FROM SMD_LH.dbo.list_columns LIMIT 1000")
display(df)

StatementMeta(, 76c7d7f5-e527-466e-a9df-ffaaaa91c563, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bd5326a1-29e7-4b50-a5e0-3a59072e6d3c)